# **Assistant Fine-Tuné (Basé sur le -Style & Ton de l'Hôtel)**

**Objectif** : Fine-tuner un modèle Mistral 7B Instruct sur la FAQ de l'Hôtel de la Promenade pour qu'il imite le style et le ton spécifiques de l'établissement.

**Contexte**
Dans les notebooks précédents, on a développé un assistant RAG capable de répondre aux questions factuelles à partir de la documentation interne. On souhaite maintenant créer un assistant conversationnel qui répond avec le style caractéristique de l'hôtel, en se basant sur sa FAQ officielle.

**Choix du modèle : Mistral 7B Instruct**
- 7,3 milliards de paramètres, excellent équilibre performance/ressources
- Licence Apache 2.0, utilisation sans restriction
- Performances solides en français et anglais
- Quantifiable en 4-bit pour fine-tuning sur T4 (Colab)

**Approche technique : QLoRA**
- Fine-tuning en 4-bit pour réduire l'empreinte mémoire (~5-6 GB VRAM)
- Adaptateurs LoRA légers (~10 MB) sauvegardés et versionnés
- Modèle de base non modifié, seule la couche d'adaptation est entraînée

**Déploiement prévu**
- Fine-tuning sur Google Colab (GPU T4 gratuit)
- Code et adaptateurs versionnés sur GitHub
- Conteneur Docker avec vLLM pour hébergement sur Koyeb (API OpenAI-compatible)

**Étapes**
1. Configuration de l'environnement (via terminal Colab)
2. Extraction et préparation de la FAQ
3. Création du dataset au format instruction/output
4. Fine-tuning avec QLoRA
5. Sauvegarde des adaptateurs (commit GitHub)
6. Tests comparatifs (modèle de base vs fine-tuné vs FAQ originale)
7. Grille d'évaluation (exactitude, ton, style, fluidité)


## **Configurations**

In [6]:
import os
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
repo_url = "https://github.com/VivanBoy/assistant-rag.git"
!git clone {repo_url}

%cd assistant-rag

Cloning into 'assistant-rag'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 152 (delta 46), reused 128 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 26.28 MiB | 25.32 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/assistant-rag/assistant-rag


In [4]:
!git switch kevs
!ls -la

Already on 'kevs'
Your branch is up to date with 'origin/kevs'.
total 16644
drwxr-xr-x 8 root root     4096 Mar  1 21:28 .
drwxr-xr-x 1 root root     4096 Mar  1 21:27 ..
-rw-r--r-- 1 root root  4635875 Mar  1 21:27 all_comments_ottawa.csv
drwxr-xr-x 2 root root     4096 Mar  1 21:28 app
drwxr-xr-x 6 root root     4096 Mar  1 21:28 data
drwxr-xr-x 8 root root     4096 Mar  1 21:29 .git
-rw-r--r-- 1 root root      369 Mar  1 21:28 .gitignore
drwxr-xr-x 4 root root     4096 Mar  1 21:28 models
drwxr-xr-x 5 root root     4096 Mar  1 21:28 notebooks
-rw-r--r-- 1 root root      319 Mar  1 21:28 pyproject.toml
-rw-r--r-- 1 root root     4600 Mar  1 21:28 requirements.txt
drwxr-xr-x 4 root root     4096 Mar  1 21:28 src
-rw-r--r-- 1 root root   609534 Mar  1 21:28 UA1_Bloc1_Analyse_Avis_Complet.ipynb
-rw-r--r-- 1 root root    26999 Mar  1 21:28 UA1_Bloc2_RAG_Notebook_Adapted_to_src_data.ipynb
-rw-r--r-- 1 root root 11710514 Mar  1 21:27 ua1_reviews_enriched.csv
-rw-r--r-- 1 root root     3631